# 🧠 Traducción Automática Shiwilu ↔ Español con SentencePiece
Este notebook entrena un modelo de traducción automático basado en `EncoderDecoderModel` usando un **tokenizer subword con SentencePiece**.

### Ventajas:
- Mejor manejo de palabras nuevas.
- Generalización más eficiente que carácter por carácter.
- Funciona bien con corpus pequeños (lenguas de escasos recursos).

In [1]:
!pip install -q transformers datasets sentencepiece sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.0 MB/s eta 0:00:00


In [10]:
def limpiar_utf8(input_path, output_path):
    with open(input_path, "rb") as infile:
        raw = infile.read()
    clean = raw.decode("utf-8", errors="ignore")
    with open(output_path, "w", encoding="utf-8") as outfile:
        outfile.write(clean)

# Aplica a ambos archivos
limpiar_utf8("shiwilu.txt", "shiwilu_utf8.txt")
limpiar_utf8("espanol.txt", "espanol_utf8.txt")


In [11]:
# Preparar textos para entrenar tokenizer conjunto
with open("shiwilu_utf8.txt", encoding="utf-8") as f1, open("espanol_utf8.txt", encoding="utf-8") as f2:
    lines = [line.strip() for line in f1.readlines()] + [line.strip() for line in f2.readlines()]

# Guardar archivo combinado
with open("combined_utf8.txt", "w", encoding="utf-8") as f:
    for line in lines:
        f.write(line + "\n")

In [12]:
# Entrenar tokenizer SentencePiece
import sentencepiece as spm
spm.SentencePieceTrainer.train(
    input='combined_utf8.txt', model_prefix='spm_shiwilu', vocab_size=500,
    model_type='unigram', pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    user_defined_symbols=['[PAD]', '[UNK]', '[BOS]', '[EOS]']
)

In [15]:
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="spm_shiwilu.model",  # a veces funciona
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[BOS]",
    eos_token="[EOS]",
)

tokenizer.model_max_length = 128


Exception: stream did not contain valid UTF-8

In [ ]:
# Preparar dataset
from datasets import Dataset

with open("shiwilu.txt", encoding="utf-8") as f1, open("espanol.txt", encoding="utf-8") as f2:
    dataset = Dataset.from_dict({
        "shw": [line.strip() for line in f1],
        "es": [line.strip() for line in f2],
    }).train_test_split(test_size=0.1)

dataset

In [ ]:
# Tokenizar datos
def preprocess(example):
    model_inputs = tokenizer(
        example['shw'], truncation=True, padding='max_length', max_length=128
    )
    labels = tokenizer(
        example['es'], truncation=True, padding='max_length', max_length=128
    ).input_ids
    model_inputs['labels'] = labels
    return model_inputs

tokenized_dataset = dataset.map(preprocess, remove_columns=['shw', 'es'])

In [ ]:
# Crear modelo encoder-decoder desde cero
from transformers import EncoderDecoderModel
model = EncoderDecoderModel.from_encoder_decoder_pretrained("bert-base-uncased", "bert-base-uncased")
model.config.pad_token_id = tokenizer.pad_token_id
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

training_args = Seq2SeqTrainingArguments(
    output_dir="./shiwilu_encoderdecoder",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    tokenizer=tokenizer
)
# trainer.train()

In [ ]:
# Función de traducción
def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(
        **inputs,
        max_length=128,
        decoder_start_token_id=tokenizer.bos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Ejemplo de prueba
print(traducir("ashintu niwan"))